<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day3(260225)_server%26html%26css%26js%EC%97%B0%EA%B2%B0_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir crp-demo
!cd crp-demo
!npm init -y
!npm install express

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}



⠙⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 65 packages, and audited 66 packages in 3s
⠼
⠼22 packages are looking for funding
⠼  run `npm fund` for details
⠼
found 0 vulnerabilities
⠼

In [ ]:
%%writefile /content/crp-demo/server.js

// server.js
const express = require("express");
const app = express();

// Blocking JS: intentionally slow (3 seconds)
app.get("/block.js", async (req, res) => {
  res.setHeader("Content-Type", "application/javascript; charset=utf-8");
  setTimeout(() => {
    res.send(`
      // Simulate heavy/slow script
      alert("Blocking Script Loaded (simulated delay)");
      console.log("block.js executed");
    `);
  }, 3000);
});

// HTML page with blocking script
app.get("/blocking", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>CRP Demo - Blocking Script</title>
</head>
<body>
  <h2>Blocking Script Demo</h2>

  <p id="before">This text is before the script tag.</p>

  <script src="/block.js"></script>

  <p id="after">This text is after the script tag (DOM continues after script executes).</p>
</body>
</html>`);
});

app.listen(3000, () => console.log("Server running on port 3000"));

Writing /content/crp-demo/server.js


In [ ]:
!nohup node /content/crp-demo/server.js > /content/crp-demo/server.log 2>&1 &

In [ ]:
!tail -n 10 crp-demo/server.log

Server running on port 3000


In [ ]:
!npm install -g cloudflared

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 1 package in 2s
⠇

In [ ]:
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &

In [ ]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log | tail -n 1

In [ ]:
%%writefile /content/crp-demo/server.js

// server.js
const express = require("express");
const app = express();

// Blocking/Deferred JS: intentionally slow (3 seconds)
app.get("/slow.js", async (req, res) => {
  res.setHeader("Content-Type", "application/javascript; charset=utf-8");
  setTimeout(() => {
    res.send(`
      console.log("slow.js executed");
      alert("slow.js executed after DOM parsing (defer test)");
    `);
  }, 3000);
});

// Blocking page: script without defer
app.get("/blocking", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>CRP Demo - Blocking Script</title>
</head>
<body>
  <h2>Blocking Script Demo</h2>
  <p id="before">This text is before the script tag.</p>

  <script src="/slow.js"></script>

  <p id="after">This text is after the script tag (may be delayed).</p>
</body>
</html>`);
});

// Defer page: script with defer (non-blocking parsing)
app.get("/defer", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>CRP Demo - Defer Script</title>
  <script src="/slow.js" defer></script>
</head>
<body>
  <h2>Defer Script Demo</h2>
  <p id="before">This text is before content.</p>
  <p id="after">This text should appear even while slow.js is downloading.</p>

  <script>
    // Observe DOM readiness timing
    document.addEventListener("DOMContentLoaded", () => {
      const el = document.createElement("p");
      el.textContent = "DOMContentLoaded fired (DOM ready)";
      document.body.appendChild(el);
    });
  </script>
</body>
</html>`);
});

app.listen(3000, () => console.log("Server running on port 3000"));

Overwriting /content/crp-demo/server.js


In [ ]:
!pkill -f "node /content/crp-demo/server.js" || true
!nohup node /content/crp-demo/server.js > /content/crp-demo/server.log 2>&1 &
!tail -n 5 /content/crp-demo/server.log

^C
Server running on port 3000


In [ ]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log | tail -n 1

https://knew-lambda-permissions-cart.trycloudflare.com


In [ ]:
!pkill -f "node /content/crp-demo/server.js" || true
!nohup node /content/crp-demo/server.js > /content/crp-demo/server.log 2>&1 &
!tail -n 5 /content/crp-demo/server.log

^C


In [ ]:
%%writefile /content/crp-demo/server.js

// server.js
const express = require("express");
const app = express();

// Slow JS (3 seconds)
app.get("/slow.js", (req, res) => {
  res.setHeader("Content-Type", "application/javascript; charset=utf-8");
  setTimeout(() => {
    res.send(`
      console.log("slow.js executed");
      alert("slow.js executed (defer test)");
    `);
  }, 3000);
});

// Slow CSS (3 seconds)
app.get("/slow.css", (req, res) => {
  res.setHeader("Content-Type", "text/css; charset=utf-8");
  setTimeout(() => {
    res.send(`
      body { font-family: Arial, sans-serif; }
      .hero { padding: 24px; border: 2px solid #333; }
      .highlight { font-size: 24px; font-weight: 700; }
    `);
  }, 3000);
});

// Blocking page
app.get("/blocking", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head><meta charset="UTF-8" /><title>Blocking Script</title></head>
<body>
  <h2>Blocking Script Demo</h2>
  <p>Before script</p>
  <script src="/slow.js"></script>
  <p>After script</p>
</body>
</html>`);
});

// Defer page
app.get("/defer", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>Defer Script</title>
  <script src="/slow.js" defer></script>
</head>
<body>
  <h2>Defer Script Demo</h2>
  <p>This should render without waiting for slow.js</p>
  <script>
    document.addEventListener("DOMContentLoaded", () => {
      const el = document.createElement("p");
      el.textContent = "DOMContentLoaded fired (DOM ready)";
      document.body.appendChild(el);
    });
  </script>
</body>
</html>`);
});

// CSS blocking page
app.get("/css-blocking", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>CSS Render Blocking</title>
  <link rel="stylesheet" href="/slow.css">
</head>
<body>
  <div class="hero">
    <div class="highlight">CSS Render-Blocking Demo</div>
    <p>If CSS is slow, first render may be delayed.</p>
  </div>
</body>
</html>`);
});

// CSS non-blocking-like comparison (critical inline + delayed external)
app.get("/css-optimized", (req, res) => {
  res.setHeader("Content-Type", "text/html; charset=utf-8");
  res.send(`<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>CSS Optimized</title>
  <style>
    body { font-family: Arial, sans-serif; }
    .hero { padding: 24px; border: 2px solid #333; }
    .highlight { font-size: 24px; font-weight: 700; }
  </style>
  <link rel="stylesheet" href="/slow.css">
</head>
<body>
  <div class="hero">
    <div class="highlight">CSS Optimized Demo</div>
    <p>Critical CSS is inline. External CSS may arrive later.</p>
  </div>
</body>
</html>`);
});

app.listen(3000, () => console.log("Server running on port 3000"));

Overwriting /content/crp-demo/server.js


In [ ]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log | tail -n 1

https://knew-lambda-permissions-cart.trycloudflare.com


In [ ]:
!pkill -f "node /content/crp-demo/server.js" || true
!nohup node /content/crp-demo/server.js > /content/crp-demo/server.log 2>&1 &
!tail -n 5 /content/crp-demo/server.log

^C


In [ ]:
!pkill -f "node /content/crp-demo/server.js" || true
!nohup node /content/crp-demo/server.js > /content/crp-demo/server.log 2>&1 &
!tail -n 5 /content/crp-demo/server.log

^C


In [ ]:
%%writefile /content/crp-demo/server.js

const express = require("express");
const path = require("path");
const app = express();

app.use(express.static(__dirname));

app.get("/slow.css", (req, res) => {
  setTimeout(() => {
    res.sendFile(path.join(__dirname, "slow.css"));
  }, 3000);
});

app.get("/block.js", (req, res) => {
  setTimeout(() => {
    res.sendFile(path.join(__dirname, "block.js"));
  }, 2000);
});

app.listen(3000, () => {
  console.log("Server running on port 3000");
});

Overwriting /content/crp-demo/server.js


In [ ]:
%%writefile /content/crp-demo/index.html

<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8" />
  <title>CRP Demo</title>
  <link rel="stylesheet" href="/slow.css">
</head>

<body>
  <h1>CRP Pipeline Demo</h1>
  <p>Observe rendering delay.</p>

  <div class="box"></div>

  <script>
    console.log("DOM parsing started");
    document.addEventListener("DOMContentLoaded", () => {
      console.log("DOMContentLoaded at", performance.now());
    });
    window.addEventListener("load", () => {
      console.log("Load event at", performance.now());
    });
  </script>

  <script src="/block.js"></script>
</body>
</html>

Overwriting /content/crp-demo/index.html


In [ ]:
%%writefile /content/crp-demo/slow.css

body {
  background: black;
  color: white;
  font-family: Arial;
}

.box {
  width: 200px;
  height: 200px;
  background: red;
}

Overwriting /content/crp-demo/slow.css


In [ ]:
%%writefile /content/crp-demo/block.js

console.log("Blocking JS executed at", performance.now());

Overwriting /content/crp-demo/block.js


In [ ]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log | tail -n 1

https://richardson-ready-garlic-without.trycloudflare.com
